## import libraries

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.pipeline import Pipeline

## load datasets

In [ ]:
train_df = pd.read_csv('data/6/train.csv')
valid_df = pd.read_csv('data/6/valid.csv')
# test_df = pd.read_csv('data/6/test.csv')

train_df.dropna(subset=['text', 'sentiment'], inplace=True)
valid_df.dropna(subset=['text', 'sentiment'], inplace=True)

## **Model Development** using Train set

- Vecotorizing (Preprocessing + Tokenization)

- Hyperparamete tuning using CV on Train set

- Re-train best model on Train set

### TF-IDF representation

In [17]:
tfidf_vectorizer = TfidfVectorizer()
model = LogisticRegression()

pipeline = Pipeline([('vectorizer', tfidf_vectorizer), ('classifier', model)])
pipeline.fit(train_df['text'], train_df['sentiment'])
predictions = pipeline.predict(train_df['text'])
print(classification_report(train_df['sentiment'], predictions))

              precision    recall  f1-score   support

    negative       0.94      0.53      0.68      1001
    positive       0.89      0.99      0.94      3911

    accuracy                           0.90      4912
   macro avg       0.92      0.76      0.81      4912
weighted avg       0.90      0.90      0.89      4912



In [ ]:
tfidf_vectorizer = TfidfVectorizer()
model = LogisticRegression(max_iter=1000)

pipeline = Pipeline([
    ('vectorizer', tfidf_vectorizer),
    ('classifier', model)
])

param_grid = {
    'vectorizer__ngram_range': [(1, 1), (1, 2)],
    'vectorizer__min_df': [1, 2, 5],
    'vectorizer__max_df': [0.9, 1.0],
    'vectorizer__max_features': [5000, 10000, 20000],
    'vectorizer__sublinear_tf': [True, False],
    'classifier__C': [0.01, 0.1, 1, 10],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['liblinear'],
    'classifier__class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(train_df['text'], train_df['sentiment'])

best_pipeline = grid_search.best_estimator_
predictions = best_pipeline.predict(train_df['text'])

print("Best parameters:")
print(grid_search.best_params_)
print("\nClassification report on train_df:")
print(classification_report(train_df['sentiment'], predictions))

Best parameters:
{'classifier__C': 10, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear', 'vectorizer__max_df': 0.9, 'vectorizer__max_features': 20000, 'vectorizer__min_df': 2, 'vectorizer__ngram_range': (1, 2), 'vectorizer__sublinear_tf': True}

Classification report on train_df:
              precision    recall  f1-score   support

    negative       0.97      0.99      0.98      1001
    positive       1.00      0.99      1.00      3911

    accuracy                           0.99      4912
   macro avg       0.98      0.99      0.99      4912
weighted avg       0.99      0.99      0.99      4912



In [20]:
valid_predictions = best_pipeline.predict(valid_df['text'])

print("Validation Accuracy:", accuracy_score(valid_df['sentiment'], valid_predictions))
print("Validation F1-macro:", f1_score(valid_df['sentiment'], valid_predictions, average='macro'))
print("\nClassification report on valid_df:")
print(classification_report(valid_df['sentiment'], valid_predictions))

Validation Accuracy: 0.8942857142857142
Validation F1-macro: 0.8398941694278226

Classification report on valid_df:
              precision    recall  f1-score   support

    negative       0.73      0.76      0.75       143
    positive       0.94      0.93      0.93       557

    accuracy                           0.89       700
   macro avg       0.83      0.85      0.84       700
weighted avg       0.90      0.89      0.90       700



Model is overfitted

In [24]:
tfidf_vectorizer = TfidfVectorizer(lowercase=True)
model = LogisticRegression(max_iter=1000)

pipeline = Pipeline([
    ('vectorizer', tfidf_vectorizer),
    ('classifier', model)
])

param_grid = {
    'vectorizer__ngram_range': [(1, 2)],
    'vectorizer__min_df': [2, 5, 10],
    'vectorizer__max_df': [0.9],
    'vectorizer__max_features': [5000, 10000],
    'vectorizer__sublinear_tf': [True],
    'classifier__C': [0.001, 0.01, 0.1, 1],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['liblinear'],
    'classifier__class_weight': ['balanced']
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(train_df['text'], train_df['sentiment'])

best_pipeline = grid_search.best_estimator_
predictions = best_pipeline.predict(train_df['text'])

print("Best parameters:")
print(grid_search.best_params_)
print("Train Accuracy:", accuracy_score(valid_df['sentiment'], valid_predictions))
print("Train F1-macro:", f1_score(valid_df['sentiment'], valid_predictions, average='macro'))
print("\nClassification report on train_df:")
print(classification_report(train_df['sentiment'], predictions))

Best parameters:
{'classifier__C': 1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear', 'vectorizer__max_df': 0.9, 'vectorizer__max_features': 10000, 'vectorizer__min_df': 2, 'vectorizer__ngram_range': (1, 2), 'vectorizer__sublinear_tf': True}
Train Accuracy: 0.8757142857142857
Train F1-macro: 0.8271649085165982

Classification report on train_df:
              precision    recall  f1-score   support

    negative       0.73      0.97      0.83      1001
    positive       0.99      0.91      0.95      3911

    accuracy                           0.92      4912
   macro avg       0.86      0.94      0.89      4912
weighted avg       0.94      0.92      0.92      4912



In [25]:
valid_predictions = best_pipeline.predict(valid_df['text'])

print("Validation Accuracy:", accuracy_score(valid_df['sentiment'], valid_predictions))
print("Validation F1-macro:", f1_score(valid_df['sentiment'], valid_predictions, average='macro'))
print("\nClassification report on valid_df:")
print(classification_report(valid_df['sentiment'], valid_predictions))

Validation Accuracy: 0.8757142857142857
Validation F1-macro: 0.8271649085165982

Classification report on valid_df:
              precision    recall  f1-score   support

    negative       0.65      0.85      0.74       143
    positive       0.96      0.88      0.92       557

    accuracy                           0.88       700
   macro avg       0.80      0.86      0.83       700
weighted avg       0.89      0.88      0.88       700



### BoW representation

In [16]:
vectorizer = CountVectorizer()
model = LogisticRegression()

pipeline = Pipeline([('vectorizer', vectorizer), ('classifier', model)])
pipeline.fit(train_df['text'], train_df['sentiment'])
predictions = pipeline.predict(train_df['text'])
print(classification_report(train_df['sentiment'], predictions))

              precision    recall  f1-score   support

    negative       0.99      0.89      0.93      1001
    positive       0.97      1.00      0.98      3911

    accuracy                           0.97      4912
   macro avg       0.98      0.94      0.96      4912
weighted avg       0.97      0.97      0.97      4912



### Word2vec representation

## **Model selection** using Validation set

## **Final Performance** using Test set